# APK bauen — für das Handy

Dieses Notizbuch baut aus dem Design-Prototyp eine installierbare **APK-Datei**.

**So geht's:** oben im Menü auf *Laufzeit → Alle ausführen*. Der erste Durchlauf
dauert etwa **10–15 Minuten**, weil das Android-SDK geladen werden muss.
Am Ende lädt die letzte Zelle die APK auf dein Gerät herunter.

Gebaut wird ein **Debug-Build**: er läuft auf jedem Gerät ab Android 5.1,
ist aber nicht für den Play Store gedacht.

> Wenn Colab meldet, dass die Sitzung abgelaufen ist: einfach neu verbinden
> und *Alle ausführen* noch einmal starten.

## 1 — Einstellungen

Hier nur etwas ändern, wenn du an einem anderen Zweig arbeitest.

In [ ]:
REPO   = 'https://github.com/todidervogel/design.git'
BRANCH = 'claude/design-spec-screens-components-omyfb5'

print('Repository:', REPO)
print('Zweig:     ', BRANCH)

## 2 — Java 17 installieren

Der Android-Baukasten braucht Java 17.

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
update-java-alternatives -s java-1.17.0-openjdk-amd64 2>/dev/null || true
java -version

In [ ]:
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']
print(os.environ['JAVA_HOME'])

## 3 — Node.js 20 installieren

In [ ]:
%%bash
set -e
curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
apt-get -qq install -y nodejs > /dev/null
node -v && npm -v

## 4 — Android-SDK installieren

Das ist der längste Schritt (etwa 1 GB Download).

In [ ]:
%%bash
set -e
mkdir -p /root/android-sdk/cmdline-tools
cd /root/android-sdk/cmdline-tools
if [ ! -d latest ]; then
  curl -fsSL -o tools.zip https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip
  unzip -q tools.zip && mv cmdline-tools latest && rm tools.zip
fi
echo 'Kommandozeilen-Werkzeuge bereit'

In [ ]:
import os
os.environ['ANDROID_HOME'] = '/root/android-sdk'
os.environ['ANDROID_SDK_ROOT'] = '/root/android-sdk'
os.environ['PATH'] = '/root/android-sdk/cmdline-tools/latest/bin:/root/android-sdk/platform-tools:' + os.environ['PATH']
print(os.environ['ANDROID_HOME'])

In [ ]:
%%bash
set -e
yes | sdkmanager --licenses > /dev/null 2>&1 || true
sdkmanager --install 'platform-tools' 'platforms;android-34' 'build-tools;34.0.0' > /dev/null
echo 'Android-SDK bereit'

## 5 — Projekt holen und Weboberfläche bauen

In [ ]:
%%bash -s "$REPO" "$BRANCH"
set -e
rm -rf /content/design
git clone --depth 1 --branch "$2" "$1" /content/design
cd /content/design
npm ci --silent
VITE_BASE=./ npm run build
echo 'Weboberfläche gebaut'

## 6 — In das Android-Projekt übernehmen und APK bauen

In [ ]:
%%bash
set -e
cd /content/design
npx cap sync android
cd android
chmod +x gradlew
./gradlew assembleDebug --no-daemon
ls -lh app/build/outputs/apk/debug/

## 7 — APK herunterladen

Die Datei landet in deinem Download-Ordner. Zum Installieren antippen und
bei der Nachfrage *Installieren aus dieser Quelle erlauben* bestätigen.

In [ ]:
import shutil
from google.colab import files

quelle = '/content/design/android/app/build/outputs/apk/debug/app-debug.apk'
ziel = '/content/tellerrand-debug.apk'
shutil.copy(quelle, ziel)
files.download(ziel)